# Model Performance Analysis: ICD Level 3, Level 4 & Phecode Schemes

Compiles and visualizes held-out test metrics across three event-coding schemes:
- **ICD Level 3** (`level_3_ICD_results/`)
- **ICD Level 4** (`level_4_ICD_results/`)
- **Phecode** (`phecode_results/`)

### Sections
1. Compile results for each scheme
2. Per-scheme analysis (text improvement, base vs text, top/bottom events)
3. Feature comparison across modalities
4. Cross-scheme comparison

In [ ]:
import os
import icd10
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

sns.set_style('whitegrid')

# === Paths ===
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
SURV_PATH = os.path.join(DATA_PATH, 'time-to-event_analysis/')
RESULTS_BASE = os.path.join(SURV_PATH, 'results/')
FIGURE_PATH = '/data/gusev/USERS/jpconnor/figures/clinical_text_embedding_project/model_metrics/'
os.makedirs(FIGURE_PATH, exist_ok=True)

FEATURES = ['stage', 'treatment', 'somatic', 'prs', 'text']

SCHEME_CONFIGS = {
    'icd3': {
        'results_dir': 'level_3_ICD_results',
        'label': 'ICD Level 3',
    },
    'icd4': {
        'results_dir': 'level_4_ICD_results',
        'label': 'ICD Level 4',
    },
    'phecode': {
        'results_dir': 'phecode_results',
        'label': 'Phecode',
    },
}

# Phecode description mapping
phecode_map_file = os.path.join(DATA_PATH, 'code_data/icd_to_phecode_map.csv')
if os.path.exists(phecode_map_file):
    _phecode_map = pd.read_csv(phecode_map_file)
    PHECODE_DESCR = dict(zip(_phecode_map['PHECODE'].astype(str), _phecode_map['PHECODE_DESCR']))
    PHECODE_CAT_DESCR = dict(zip(_phecode_map['PHECODE'].astype(str), _phecode_map['PHECODE_CAT_DESCR']))
else:
    print(f'WARNING: phecode mapping not found at {phecode_map_file}')
    PHECODE_DESCR = {}
    PHECODE_CAT_DESCR = {}

## 1. Compile Results

In [ ]:
def get_event_description(event, scheme):
    """Look up event description based on scheme type."""
    if scheme in ('icd3', 'icd4'):
        if icd10.exists(event):
            code = icd10.find(event)
            return code.description
    elif scheme == 'phecode':
        return PHECODE_DESCR.get(event)
    return None


def get_event_category(event, scheme):
    """Look up event category/chapter based on scheme type."""
    if scheme in ('icd3', 'icd4'):
        if icd10.exists(event):
            code = icd10.find(event)
            try:
                return code.block_description
            except Exception:
                return None
    elif scheme == 'phecode':
        return PHECODE_CAT_DESCR.get(event)
    return None


def compile_scheme_results(scheme):
    """Compile full-cohort and feature-comp test metrics for a given scheme."""
    cfg = SCHEME_CONFIGS[scheme]
    results_path = os.path.join(RESULTS_BASE, cfg['results_dir'])
    full_cohort_path = os.path.join(results_path, 'full_cohort/')
    feature_comps_path = os.path.join(results_path, 'feature_comps/')

    full_events = set(os.listdir(full_cohort_path)) if os.path.isdir(full_cohort_path) else set()
    feat_events = set(os.listdir(feature_comps_path)) if os.path.isdir(feature_comps_path) else set()
    events = sorted(full_events & feat_events)
    print(f'[{cfg["label"]}] {len(events)} events with both full_cohort and feature_comps results')

    rows = []
    for event in events:
        event_full = os.path.join(full_cohort_path, event)
        event_feat = os.path.join(feature_comps_path, event)

        # Full cohort metrics
        try:
            text_test = pd.read_csv(os.path.join(event_full, 'text_test.csv'))
            base_test = pd.read_csv(os.path.join(event_full, 'type_model_metrics.csv'))
            base_test = base_test.loc[base_test['eval_data'] == 'test_data']
        except Exception:
            continue

        base_c = base_test['mean_c_index'].values[0]
        base_auc = base_test['mean_auc(t)'].values[0]
        text_c = text_test['mean_c_index'].values[0]
        text_auc = text_test['mean_auc(t)'].values[0]

        # Feature comp metrics
        feat_c = []
        feat_auc = []
        for feat in FEATURES:
            try:
                feat_df = pd.read_csv(os.path.join(event_feat, f'{feat}_test.csv'))
                feat_c.append(feat_df['mean_c_index'].values[0])
                feat_auc.append(feat_df['mean_auc(t)'].values[0])
            except Exception:
                feat_c.append(np.nan)
                feat_auc.append(np.nan)

        descr = get_event_description(event, scheme)
        category = get_event_category(event, scheme)

        rows.append({
            'event': event,
            'event_description': descr,
            'event_category': category,
            'base_c_index': base_c,
            'text_c_index': text_c,
            'base_auc': base_auc,
            'text_auc': text_auc,
            **{f'{f}_c_index': c for f, c in zip(FEATURES, feat_c)},
            **{f'{f}_auc': a for f, a in zip(FEATURES, feat_auc)},
        })

    df = pd.DataFrame(rows)
    df['delta_c_index'] = df['text_c_index'] - df['base_c_index']
    df['delta_auc'] = df['text_auc'] - df['base_auc']
    return df

In [ ]:
scheme_dfs = {}
for scheme in SCHEME_CONFIGS:
    scheme_dfs[scheme] = compile_scheme_results(scheme)
    print(f'  Compiled {len(scheme_dfs[scheme])} events\n')

## 2. Per-Scheme Analysis

In [ ]:
def plot_scheme_summary(df, label, metric='c_index'):
    """Generate a 2x2 summary figure for a single scheme."""
    base_col = f'base_{metric}'
    text_col = f'text_{metric}'
    delta_col = f'delta_{metric}'
    metric_label = 'C-index' if metric == 'c_index' else 'Mean AUC(t)'

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    fig.suptitle(f'{label} — {metric_label}', fontsize=14, fontweight='bold')

    # (0,0) Histogram of improvement
    ax = axes[0, 0]
    mean_delta = df[delta_col].mean()
    sns.histplot(df[delta_col].dropna(), ax=ax, bins=25)
    ax.axvline(mean_delta, color='red', ls='--', label=f'Mean = {mean_delta:.3f}')
    ax.set_xlabel(f'Text − Base ({metric_label})')
    ax.set_title('Distribution of Text Improvement')
    ax.legend()

    # (0,1) Scatter: base vs text
    ax = axes[0, 1]
    ax.scatter(df[base_col], df[text_col], alpha=0.5, s=20)
    lims = [min(df[base_col].min(), df[text_col].min()) - 0.02,
            max(df[base_col].max(), df[text_col].max()) + 0.02]
    ax.plot(lims, lims, 'r--', alpha=0.7)
    ax.set_xlabel(f'Base {metric_label}')
    ax.set_ylabel(f'Text {metric_label}')
    ax.set_title('Base vs Text Model')

    # (1,0) Top 15 improvements
    ax = axes[1, 0]
    top = df.nlargest(15, delta_col)
    labels = [f"{r['event']}" + (f" ({r['event_description'][:30]}...)" if pd.notna(r['event_description']) and len(str(r['event_description'])) > 30
              else f" ({r['event_description']})" if pd.notna(r['event_description']) else '')
             for _, r in top.iterrows()]
    ax.barh(range(len(top)), top[delta_col].values, color='steelblue')
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(f'Δ {metric_label}')
    ax.set_title('Top 15 Improvements with Text')

    # (1,1) Bottom 15 (worst degradations)
    ax = axes[1, 1]
    bottom = df.nsmallest(15, delta_col)
    labels = [f"{r['event']}" + (f" ({r['event_description'][:30]}...)" if pd.notna(r['event_description']) and len(str(r['event_description'])) > 30
              else f" ({r['event_description']})" if pd.notna(r['event_description']) else '')
             for _, r in bottom.iterrows()]
    ax.barh(range(len(bottom)), bottom[delta_col].values, color='indianred')
    ax.set_yticks(range(len(bottom)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(f'Δ {metric_label}')
    ax.set_title('Bottom 15 (Degraded with Text)')

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, f'{label.replace(" ", "_").lower()}_{metric}_summary.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for scheme, cfg in SCHEME_CONFIGS.items():
    df = scheme_dfs[scheme]
    if len(df) == 0:
        print(f'No results for {cfg["label"]}, skipping.')
        continue
    plot_scheme_summary(df, cfg['label'], metric='c_index')
    plot_scheme_summary(df, cfg['label'], metric='auc')

## 3. Feature Comparison Across Modalities

In [ ]:
def plot_feature_comparison(df, label, metric='c_index'):
    """Compare feature modality performance for a scheme."""
    metric_label = 'C-index' if metric == 'c_index' else 'Mean AUC(t)'
    feat_cols = [f'{f}_{metric}' for f in FEATURES]

    # Melt to long format
    long = df.melt(id_vars=['event'], value_vars=feat_cols,
                   var_name='modality', value_name=metric_label)
    long['modality'] = long['modality'].str.replace(f'_{metric}', '')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'{label} — Feature Comparison ({metric_label})', fontsize=14, fontweight='bold')

    # Box plot
    ax = axes[0]
    sns.boxplot(data=long, x='modality', y=metric_label, order=FEATURES, ax=ax)
    ax.set_title(f'{metric_label} by Feature Modality')
    ax.set_xlabel('')

    # Mean bar plot with SD
    ax = axes[1]
    means = long.groupby('modality')[metric_label].agg(['mean', 'std']).reindex(FEATURES)
    ax.bar(means.index, means['mean'], yerr=means['std'], capsize=4, color='steelblue', alpha=0.8)
    ax.set_ylabel(f'Mean {metric_label}')
    ax.set_title(f'Mean ± SD Across Events')
    ax.set_xlabel('')

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURE_PATH, f'{label.replace(" ", "_").lower()}_{metric}_feature_comp.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for scheme, cfg in SCHEME_CONFIGS.items():
    df = scheme_dfs[scheme]
    if len(df) == 0:
        continue
    plot_feature_comparison(df, cfg['label'], metric='c_index')
    plot_feature_comparison(df, cfg['label'], metric='auc')

## 4. Cross-Scheme Comparison

In [ ]:
# Aggregate statistics per scheme
summary_rows = []
for scheme, cfg in SCHEME_CONFIGS.items():
    df = scheme_dfs[scheme]
    if len(df) == 0:
        continue
    summary_rows.append({
        'scheme': cfg['label'],
        'n_events': len(df),
        'mean_base_c_index': df['base_c_index'].mean(),
        'mean_text_c_index': df['text_c_index'].mean(),
        'mean_delta_c_index': df['delta_c_index'].mean(),
        'median_delta_c_index': df['delta_c_index'].median(),
        'pct_improved': (df['delta_c_index'] > 0).mean() * 100,
        'mean_base_auc': df['base_auc'].mean(),
        'mean_text_auc': df['text_auc'].mean(),
        'mean_delta_auc': df['delta_auc'].mean(),
        'median_delta_auc': df['delta_auc'].median(),
        'pct_improved_auc': (df['delta_auc'] > 0).mean() * 100,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.round(4)

In [ ]:
# Distribution of text improvement across schemes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for metric, metric_label, ax in [('c_index', 'C-index', axes[0]), ('auc', 'AUC(t)', axes[1])]:
    delta_col = f'delta_{metric}'
    for scheme, cfg in SCHEME_CONFIGS.items():
        df = scheme_dfs[scheme]
        if len(df) == 0:
            continue
        sns.kdeplot(df[delta_col].dropna(), ax=ax, label=f"{cfg['label']} (n={len(df)})")
    ax.axvline(0, color='grey', ls=':', alpha=0.7)
    ax.set_xlabel(f'Δ {metric_label} (Text − Base)')
    ax.set_title(f'Text Improvement Distribution ({metric_label})')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_PATH, 'cross_scheme_improvement_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-scheme feature modality comparison
feat_summary = []
for scheme, cfg in SCHEME_CONFIGS.items():
    df = scheme_dfs[scheme]
    if len(df) == 0:
        continue
    for feat in FEATURES:
        col = f'{feat}_c_index'
        feat_summary.append({
            'scheme': cfg['label'],
            'modality': feat,
            'mean_c_index': df[col].mean(),
            'std_c_index': df[col].std(),
        })

feat_summary_df = pd.DataFrame(feat_summary)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=feat_summary_df, x='modality', y='mean_c_index', hue='scheme',
            order=FEATURES, ax=ax)
ax.set_ylabel('Mean C-index')
ax.set_xlabel('')
ax.set_title('Feature Modality Performance Across Schemes')
ax.legend(title='Scheme')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_PATH, 'cross_scheme_feature_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Save Compiled Results

In [ ]:
for scheme, cfg in SCHEME_CONFIGS.items():
    df = scheme_dfs[scheme]
    if len(df) == 0:
        continue
    results_path = os.path.join(RESULTS_BASE, cfg['results_dir'])
    c_index_file = os.path.join(results_path, f'{cfg["results_dir"]}_c_index_df.csv')
    auc_file = os.path.join(results_path, f'{cfg["results_dir"]}_mean_auc_df.csv')
    df.to_csv(c_index_file, index=False)
    print(f'Saved {c_index_file}')